# RAG Q&A Chatbot for Course Content

A retrieval-augmented generation (RAG) tutor built with **LangChain**. A student asks a question about a video course; the bot retrieves the most relevant transcript excerpts from a **Chroma** vector store and answers **only** from those excerpts, naming the lecture it used. Questions outside the course are declined instead of answered from model memory.

| Component | Choice |
| --- | --- |
| Orchestration | LangChain Expression Language (LCEL) |
| Chat model | `openai/gpt-oss-120b` (OpenAI open-weight model) served by Groq via `ChatGroq` |
| Embeddings | `sentence-transformers/all-MiniLM-L6-v2` via `HuggingFaceEmbeddings` (local, 384 dims) |
| Vector store | Chroma (persisted, similarity search, k = 4) |
| Data | 25-lecture course transcript (PDF with `#` section and `##` lecture headers) |

## Pipeline

1. **Load** the transcript with `PyPDFLoader`.
2. **Split by structure** with `MarkdownHeaderTextSplitter` so every lecture keeps its section and title as metadata.
3. **Clean** speech-to-text errors with an LLM correction chain (cached to disk).
4. **Chunk** each lecture with `TokenTextSplitter` (120 tokens, 20 overlap).
5. **Embed and index** the chunks in Chroma.
6. **Prompt** with a grounded system message: answer only from the excerpts.
7. **Chain v1**: retriever + prompt + model (raw `Document` context, kept for comparison).
8. **Chain v2**: a `RunnableLambda` formats numbered excerpts with section and lecture labels.
9. **Stream** the answer token by token.

Cells marked **Local test** run without calling the LLM API. Cells marked **API test** call it.

## 0. Setup

Install the dependencies from `requirements.txt` (or run the cell below), copy `.env.example` to `.env`, and add your `GROQ_API_KEY`. Put the course transcript PDF in `data/` (see `data/README.md`). The first run downloads the embedding model (about 90 MB).

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json
import os
import shutil
from pathlib import Path

from dotenv import load_dotenv

# Paths are relative to the repository root.
ROOT = Path.cwd()
ENV_PATH = ROOT / ".env"
PDF_PATH = ROOT / "data" / "Introduction_to_Tableau.pdf"
CACHE_PATH = ROOT / "data" / "corrected_lectures.json"
PERSIST_DIR = str(ROOT / "chroma_intro_tableau")

GROQ_MODEL = "openai/gpt-oss-120b"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = 384  # must match EMBED_MODEL
CHUNK_SIZE = 120
CHUNK_OVERLAP = 20
EXPECTED_LECTURES = 25
RETRIEVER_K = 4

load_dotenv(ENV_PATH, override=True)
assert os.getenv("GROQ_API_KEY"), "Set GROQ_API_KEY in .env (see .env.example)."
assert PDF_PATH.exists(), f"Put the course transcript at {PDF_PATH.relative_to(ROOT)}"

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

print("GROQ_API_KEY: loaded")
print("Environment check passed.")

## 1. Load the course transcript

`PyPDFLoader` returns one `Document` per page. Pages are joined with newlines so the `#` and `##` headers stay on their own lines; collapsing whitespace would destroy the structure the next step splits on.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()

print("Pages:", len(pages))
print("First page metadata:", pages[0].metadata)
print()
print(pages[0].page_content[:700])


Pages: 49
First page metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-10-11T11:01:10+03:00', 'author': 'python-docx', 'moddate': '2024-10-11T11:01:10+03:00', 'source': 'Introduction_to_Tableau.pdf', 'total_pages': 49, 'page': 0, 'page_label': '1'}

# Introduction to Tableau 
## Welcome to Tableau 
Hi, everyone. 
I'm Ned and I'll be your instructor for this 
course. 
Tableau is an invaluable tool. 
One needs to learn on their journey to become a 
successful business intelligence analyst or 
data scientist. 
The art of these professions is storytelling 
using data to tell stories and convince top 
management of the right course of action. 
By completing this part of the program, you 
will know how to create charts and dashboards 
in tableaux. 
This is an essential step on your way to a data 
scientist role. 
 
## Why use Tableau: Make your data make an impact 
Tableau has grown to become one of the most 
pop

In [ ]:
# Local test. No Groq call.
transcript = "\n".join(page.page_content for page in pages)

assert len(pages) >= 1, "The PDF loader returned no pages."
assert len(transcript) > 1000, "The transcript is too short to be the course PDF."
assert "# Introduction to Tableau" in transcript
assert "## Welcome to Tableau" in transcript
assert transcript.count("\n## ") >= EXPECTED_LECTURES

print("Characters:", len(transcript))
print("Part 1 passed.")


Characters: 72734
Part 1 passed.


## 2. Split by lecture with `MarkdownHeaderTextSplitter`

`#` becomes `Course Section` and `##` becomes `Lecture Title` in each document's metadata. The header line is stripped from the body, so the correction chain rewrites lecture text only while the labels travel with every later chunk. Expected result: **25 lectures**.

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Course Section"),
    ("##", "Lecture Title"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=True,
)

lectures = markdown_splitter.split_text(transcript)

for lecture in lectures:
    print(
        f"{lecture.metadata.get('Course Section')}  |  "
        f"{lecture.metadata.get('Lecture Title')}  |  "
        f"{len(lecture.page_content)} chars"
    )


Introduction to Tableau  |  Welcome to Tableau  |  503 chars
Introduction to Tableau  |  Why use Tableau: Make your data make an impact  |  3414 chars
Introduction to Tableau  |  Let's download Tableau Public  |  1477 chars
Introduction to Tableau  |  Connecting data in Tableau  |  1647 chars
Introduction to Tableau  |  Exploring Tableau's interface  |  7020 chars
Introduction to Tableau  |  Let's create our first chart in Tableau!  |  5039 chars
Tableau Functionalities  |  Creating a table  |  3839 chars
Tableau Functionalities  |  Creating custom fields  |  1931 chars
Tableau Functionalities  |  Creating a custom field and adding calculations to a table  |  2748 chars
Tableau Functionalities  |  Adding totals and subtotals  |  1417 chars
Tableau Functionalities  |  Adding a custom calculation  |  2194 chars
Tableau Functionalities  |  Inserting a filter  |  1707 chars
Tableau Functionalities  |  Working with joins in Tableau  |  3599 chars
The Tableau Exercise  |  Introduction to the

In [ ]:
# Local test. No Groq call.
titles = [lecture.metadata.get("Lecture Title") for lecture in lectures]

assert len(lectures) == EXPECTED_LECTURES, len(lectures)
assert all(titles), "A lecture is missing its title metadata."
assert len(set(titles)) == EXPECTED_LECTURES, titles
assert all(lecture.page_content.strip() for lecture in lectures)
assert all(not lecture.page_content.lstrip().startswith("#") for lecture in lectures)

print("Part 2 passed.", len(titles), "lectures")

Part 2 passed. 25 lectures


## 3. LLM chain that cleans the transcript

The source is a speech-to-text transcript (`work book`, `dash board`, `dimenshuns`, `calclated`). A short LCEL chain rewrites one lecture at a time under strict rules: fix recognition errors, keep formulas character for character, add no new facts.

```text
Document -> RunnableLambda(prepare_lecture) -> prompt -> ChatGroq -> StrOutputParser -> RunnableLambda(strip_fences)
```

Temperature is 0 with a fixed seed. Results are cached to `data/corrected_lectures.json`, so the API is called once per lecture; delete the file to force a fresh rewrite.

In [ ]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_groq import ChatGroq


def make_chat(max_tokens: int) -> ChatGroq:
    return ChatGroq(
        model=GROQ_MODEL,
        model_kwargs={"seed": 365},
        temperature=0,
        max_tokens=max_tokens,
    )


correction_llm = make_chat(max_tokens=120000)

correction_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You edit one lecture from the Introduction to Tableau course. "
        "The text is a speech-to-text transcript, so some Tableau words are misspelled or split apart. "
        "Rewrite it as clear spoken teaching prose.\n"
        "Rules:\n"
        "1. Do not add facts, menu paths, or formulas that are not already in the transcript.\n"
        "2. Fix recognition errors. Examples: work book -> workbook, work sheet -> worksheet, "
        "dash board -> dashboard, dimenshuns -> dimensions, measurs -> measures, "
        "calclated -> calculated, pil -> pill, jine -> join.\n"
        "3. Rejoin words that were broken only by a line wrap.\n"
        "4. Keep formulas character for character, including SUM([Profit]) / SUM([Sales]) and [Sales] - [Cost].\n"
        "5. Return only the corrected lecture body. No title line, no markdown header, no bullets you were not given, no code fence.",
    ),
    (
        "human",
        "Course section: {course_section}\n"
        "Lecture title: {lecture_title}\n\n"
        "Transcript:\n{transcript}",
    ),
])

correction_parser = StrOutputParser()
correction_body = correction_prompt | correction_llm | correction_parser


def prepare_lecture(lecture: Document) -> dict:
    return {
        "course_section": lecture.metadata.get("Course Section", ""),
        "lecture_title": lecture.metadata.get("Lecture Title", ""),
        "transcript": lecture.page_content,
    }


def strip_fences(text: str) -> str:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[-1]
        if "```" in cleaned:
            cleaned = cleaned[: cleaned.rfind("```")]
    return cleaned.strip()


# Regular functions, explicitly wrapped. This is the form the project asks you to build.
correct_one_lecture = (
    RunnableLambda(prepare_lecture)
    | correction_body
    | RunnableLambda(strip_fences)
)

print(correct_one_lecture)


In [ ]:
# API test. One lecture only.
sample = lectures[0]
print("BEFORE")
print(sample.page_content[:700])
print("\nAFTER")

corrected_sample = correct_one_lecture.invoke(sample)
print(corrected_sample[:700])

assert isinstance(corrected_sample, str)
assert len(corrected_sample) > 200
assert not corrected_sample.lstrip().startswith("#")
assert "```" not in corrected_sample

print("\nPart 3 single-lecture test passed.")


BEFORE
Hi, everyone.
I'm Ned and I'll be your instructor for this
course.
Tableau is an invaluable tool.
One needs to learn on their journey to become a
successful business intelligence analyst or
data scientist.
The art of these professions is storytelling
using data to tell stories and convince top
management of the right course of action.
By completing this part of the program, you
will know how to create charts and dashboards
in tableaux.
This is an essential step on your way to a data
scientist role.

AFTER
Hi, everyone. I'm Ned, and I'll be your instructor for this course. Tableau is an invaluable tool that you need to learn on your journey to become a successful business intelligence analyst or data scientist. The art of these professions is storytelling—using data to tell stories and convince top management of the right course of action. By completing this part of the program, you will know how to create charts and dashboards in Tableau. This is an essential step on your way to

In [ ]:
import time

from langchain_core.documents import Document


def invoke_with_retry(lecture: Document, attempts: int = 3) -> str:
    """Correct one lecture, backing off on transient API errors."""
    for attempt in range(1, attempts + 1):
        try:
            return correct_one_lecture.invoke(lecture)
        except Exception as exc:  # rate limits, timeouts
            if attempt == attempts:
                raise
            wait = 2 ** attempt
            print(f"  retry {attempt}/{attempts - 1} in {wait}s ({type(exc).__name__})")
            time.sleep(wait)


if CACHE_PATH.exists():
    cached = json.loads(CACHE_PATH.read_text(encoding="utf-8"))
    corrected_lectures = [Document(page_content=item["page_content"], metadata=item["metadata"]) for item in cached]
    print(f"Loaded {len(corrected_lectures)} corrected lectures from cache.")
else:
    corrected_lectures = []
    for index, lecture in enumerate(lectures, start=1):
        title = lecture.metadata.get("Lecture Title")
        print(f"Correcting {index}/{len(lectures)}: {title}")
        try:
            text = invoke_with_retry(lecture)
        except Exception as exc:
            print(f"  failed ({type(exc).__name__}); keeping the original transcript")
            text = ""
        # Fall back to the original text if the model returns nothing.
        text = text.strip() or lecture.page_content.strip()
        corrected_lectures.append(Document(page_content=text, metadata=dict(lecture.metadata)))
        print(f"  saved {index}/{len(lectures)}")

    payload = [{"page_content": d.page_content, "metadata": d.metadata} for d in corrected_lectures]
    CACHE_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")

# Local test. No API call.
assert len(corrected_lectures) == EXPECTED_LECTURES
assert all(d.page_content.strip() for d in corrected_lectures)
assert [d.metadata.get("Lecture Title") for d in corrected_lectures] == titles
print("Part 3 passed.")

## 4. Chunk each lecture with `TokenTextSplitter`

Lectures are too long to use as retrieval units. Each one is cut into 120-token windows with a 20-token overlap (`cl100k_base` encoding), so a sentence on a boundary is visible in both neighbours. `split_documents` copies the lecture metadata onto every chunk, which is what lets the tutor cite its source. A `CharacterTextSplitter` run is shown for comparison only.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter, TokenTextSplitter

token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

chunks = token_splitter.split_documents(corrected_lectures)

character_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=500,
    chunk_overlap=50,
)
character_chunks = character_splitter.split_documents(corrected_lectures)

print("Token chunks:", len(chunks))
print("Character chunks (comparison only):", len(character_chunks))
print("First chunk metadata:", chunks[0].metadata)
print()
print(chunks[0].page_content[:400])


Token chunks: 133
Character chunks (comparison only): 156
First chunk metadata: {'Course Section': 'Introduction to Tableau', 'Lecture Title': 'Welcome to Tableau'}

Hi, everyone. I'm Ned, and I'll be your instructor for this course. Tableau is an invaluable tool that you need to learn on your journey to become a successful business intelligence analyst or data scientist. The art of these professions is storytelling—using data to tell stories and convince top management of the right course of action. By completing this part of the program, you will know how to


In [ ]:
from collections import Counter

for index, lecture in enumerate(corrected_lectures):
    text = lecture.page_content.strip() or lectures[index].page_content.strip()
    corrected_lectures[index] = Document(
        page_content=text,
        metadata=dict(lectures[index].metadata),
    )

chunks = []
for lecture in corrected_lectures:
    pieces = [
        piece
        for piece in token_splitter.split_documents([lecture])
        if piece.page_content.strip()
    ]
    if not pieces:
        print("Kept whole lecture:", lecture.metadata.get("Lecture Title"))
        pieces = [lecture]
    chunks.extend(pieces)

counts = Counter(chunk.metadata.get("Lecture Title") for chunk in chunks)
missing = sorted(set(titles) - set(counts))
extra = sorted(set(counts) - set(titles))
print("Missing:", missing)
print("Extra:", extra)

assert len(chunks) > len(corrected_lectures), len(chunks)
assert all(chunk.page_content.strip() for chunk in chunks)
assert all(chunk.metadata.get("Lecture Title") for chunk in chunks)
assert all(chunk.metadata.get("Course Section") for chunk in chunks)
assert not missing and not extra

print("Chunks per lecture:")
for title, count in counts.items():
    print(f"  {count:2d}  {title}")
print("Part 4 passed. Total chunks:", len(chunks))

Missing: []
Extra: []
Chunks per lecture:
   1  Welcome to Tableau
   4  Why use Tableau: Make your data make an impact
   1  Let's download Tableau Public
   4  Connecting data in Tableau
  12  Exploring Tableau's interface
  10  Let's create our first chart in Tableau!
   8  Creating a table
   4  Creating custom fields
   6  Creating a custom field and adding calculations to a table
   4  Adding totals and subtotals
   6  Adding a custom calculation
   4  Inserting a filter
   7  Working with joins in Tableau
   4  Introduction to the exercise
   3  Let's create a dashboard - Visualizing the three charts we want to create
   8  Using Joins in Tableau
   6  Performing a Numbers Check - Attempt #1
   9  Blending Data in Tableau
   6  Performing a Numbers Check - Attempt #2
   6  First chart
   5  Second chart
   5  Third Chart
   3  Creating and Formatting a Dashboard
   7  Adding Interactive Filters for Improved Analysis
   3  Interactive filters - fix
Part 4 passed. Total chunks: 13

## 5. Embeddings, vector store and retriever

Chunks are embedded locally with `all-MiniLM-L6-v2` (no API cost) and stored in a persisted Chroma collection. The folder is rebuilt on each run so chunks are never indexed twice. The retriever uses similarity search with k = 4; maximal marginal relevance (MMR) is left as a commented alternative for near-duplicate hits.

In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

probe = embedding.embed_query("dimensions and measures")
assert isinstance(probe, list) and len(probe) == EMBEDDING_DIM, len(probe)
print("Embedding dimension:", len(probe))

persist_path = Path(PERSIST_DIR)
if persist_path.exists():
    shutil.rmtree(persist_path)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory=PERSIST_DIR,
    collection_name="intro_tableau",
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": RETRIEVER_K},
)

# Alternative when the top chunks repeat the same paragraph:
# retriever = vectorstore.as_retriever(
#     search_type="mmr",
#     search_kwargs={"k": RETRIEVER_K, "fetch_k": 12},
# )

print("Collection count:", vectorstore._collection.count())
print("Retriever:", retriever)


Embedding dimension: 384
Collection count: 136
Retriever: tags=['Chroma', 'HuggingFaceEmbeddings'] vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000002387CB72DB0> search_kwargs={'k': 4}


In [ ]:
# Local test. Embeds the question. Does not call Groq.
question = "What is the difference between a join and data blending?"
hits = retriever.invoke(question)

assert len(hits) == RETRIEVER_K
assert all(isinstance(hit, Document) for hit in hits)
assert all(hit.page_content.strip() for hit in hits)

blob = " ".join(hit.page_content.lower() for hit in hits)
assert "blend" in blob or "join" in blob, blob[:400]

print("Top lectures for the join/blend question:")
for hit in hits:
    print(" -", hit.metadata.get("Lecture Title"))

scored = vectorstore.similarity_search_with_score(question, k=RETRIEVER_K)
print("\nDistances (smaller is closer for this store):")
for hit, score in scored:
    print(f"  {score:7.4f}  {hit.metadata.get('Lecture Title')}")

print("Part 5 passed.")


Top lectures for the join/blend question:
 - Blending Data in Tableau
 - Blending Data in Tableau
 - Blending Data in Tableau
 - Blending Data in Tableau

Distances (smaller is closer for this store):
   0.4584  Blending Data in Tableau
   0.5927  Blending Data in Tableau
   0.6584  Blending Data in Tableau
   0.7264  Blending Data in Tableau
Part 5 passed.


## 6. Grounded prompt template

The **system** message restricts the tutor to the supplied excerpts and tells it to say when the course does not cover a topic. The **human** message carries `{context}` and `{question}`. The test renders the messages for a fake excerpt without calling the model.

In [ ]:
qa_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a teaching assistant for the Introduction to Tableau course by Ned Krastev. "
        "Answer the student using only the course excerpts. "
        "If the excerpts do not contain the answer, say that this course does not cover the topic. "
        "Do not invent menu paths, product limits, or formulas. "
        "Be concrete and short. Name the lecture when that helps the student check the source. "
        "Do not start with a label such as 'Answer:'.",
    ),
    (
        "human",
        "Course excerpts:\n{context}\n\nStudent question:\n{question}",
    ),
])

preview = qa_prompt.invoke({
    "context": "Lecture: Dimensions, Measures, and Pills\nBlue pills are discrete. Green pills are continuous.",
    "question": "What does a blue pill mean in Tableau?",
})

for message in preview.messages:
    print(type(message).__name__)
    print(message.content)
    print("---")

assert len(preview.messages) == 2
assert preview.messages[0].type == "system"
assert preview.messages[1].type == "human"
assert "blue pill" in preview.messages[1].content.lower()
print("Part 6 passed.")


SystemMessage
You are a teaching assistant for the Introduction to Tableau course by Ned Krastev. Answer the student using only the course excerpts. If the excerpts do not contain the answer, say that this course does not cover the topic. Do not invent menu paths, product limits, or formulas. Be concrete and short. Name the lecture when that helps the student check the source. Do not start with a label such as 'Answer:'.
---
HumanMessage
Course excerpts:
Lecture: Dimensions, Measures, and Pills
Blue pills are discrete. Green pills are continuous.

Student question:
What does a blue pill mean in Tableau?
---
Part 6 passed.


## 7. Q&A chain, version 1

`RunnableParallel` sends the same question to the retriever and to a `RunnablePassthrough`, then into the prompt, the model and a `StrOutputParser`. This version passes the raw list of `Document` objects as context; the output below shows why that is noisy.

```text
question -> { context: retriever, question: passthrough } -> prompt -> ChatGroq -> str
```

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

chat = make_chat(max_tokens=600)

qa_chain_v1 = (
    RunnableParallel(
        context=retriever,
        question=RunnablePassthrough(),
    )
    | qa_prompt
    | chat
    | StrOutputParser()
)

# Local test: render what version 1 would send, without calling Groq.
v1_question = "What is a calculated field?"
v1_docs = retriever.invoke(v1_question)
v1_messages = qa_prompt.invoke({"context": v1_docs, "question": v1_question})
v1_context = v1_messages.messages[1].content

assert "Document(" in v1_context or "page_content" in v1_context
print(v1_context[:900])
print("\nPart 7 wiring passed. Invoke qa_chain_v1 only if you want to compare a noisy answer.")


Course excerpts:
[Document(id='1b4d9247-0eb9-4bdc-8e7c-5cb11aaed25c', metadata={'Lecture Title': 'Creating a custom field and adding calculations to a table', 'Course Section': 'Tableau Functionalities'}, page_content=' Records field. Each row represents a single car sale, so if we multiply the Price field by the Number of Records field, we’ll obtain revenue. Do you agree? Great, let’s do that.\n\nWe already know how to create a calculated field. I’ll name it Revenue and take advantage of Tableau’s autocomplete suggestions by multiplying Price and Number of Records. As we mentioned in the previous lesson, we can use the star operator (*) to carry out a multiplication. One very useful feature you’ve probably noticed is that at the bottom of the screen Tableau tells us whether the calculation we’ve entered'), Document(id='fb91083a-848f-4fbb-b57f-c3fa030f0371', metadata={'Lecture Title': 'C

Part 7 wiring passed. Invoke qa_chain_v1 only if you want to compare a noisy answer.


## 8. Q&A chain, version 2: formatted context

`format_context`, wrapped in `RunnableLambda`, turns the retrieved documents into numbered excerpts labelled with section and lecture. A second lambda strips whitespace from the question before retrieval.

```text
question -> strip -> { context: retriever | format_context, question: passthrough } -> prompt -> ChatGroq -> str
```

The unit test uses two fake documents, so it needs neither Chroma nor the API.

In [ ]:
def format_context(docs: list[Document]) -> str:
    """Join retrieved documents into one prompt block."""
    blocks = []
    for index, doc in enumerate(docs, start=1):
        section = doc.metadata.get("Course Section", "Unknown section")
        lecture = doc.metadata.get("Lecture Title", "Unknown lecture")
        blocks.append(
            f"Excerpt {index}\n"
            f"Section: {section}\n"
            f"Lecture: {lecture}\n"
            f"{doc.page_content.strip()}"
        )
    return "\n\n".join(blocks)


format_context_runnable = RunnableLambda(format_context)
normalize_question = RunnableLambda(lambda question: question.strip())

qa_chain = (
    normalize_question
    | RunnableParallel(
        context=retriever | format_context_runnable,
        question=RunnablePassthrough(),
    )
    | qa_prompt
    | chat
    | StrOutputParser()
)


In [ ]:
# Local test. No Groq call.
fake_docs = [
    Document(
        page_content="Blue pills are discrete.",
        metadata={
            "Course Section": "Connecting and Preparing Data",
            "Lecture Title": "Dimensions, Measures, and Pills",
        },
    ),
    Document(
        page_content="A dashboard combines several worksheets.",
        metadata={
            "Course Section": "Getting Started with Tableau",
            "Lecture Title": "What Tableau Is",
        },
    ),
]

formatted = format_context(fake_docs)
assert format_context_runnable.invoke(fake_docs) == formatted
assert "Lecture: Dimensions, Measures, and Pills" in formatted
assert "Blue pills are discrete." in formatted
assert formatted.index("Pills") < formatted.index("What Tableau Is")
assert normalize_question.invoke("  hello  ") == "hello"
assert hasattr(qa_chain, "stream") and hasattr(qa_chain, "invoke")

clean_messages = qa_prompt.invoke({
    "context": formatted,
    "question": "What does a blue pill mean?",
})
clean_context = clean_messages.messages[1].content
assert "Document(" not in clean_context
assert "Excerpt 1" in clean_context

print(formatted)
print("\nPart 8 passed.")


Excerpt 1
Section: Connecting and Preparing Data
Lecture: Dimensions, Measures, and Pills
Blue pills are discrete.

Excerpt 2
Section: Getting Started with Tableau
Lecture: What Tableau Is
A dashboard combines several worksheets.

Part 8 passed.


## 9. Stream the answer

`qa_chain.stream` yields text as it is generated. The last demo question is deliberately outside the course: a grounded tutor should decline rather than invent an answer.

In [ ]:
def ask(question: str) -> str:
    print("Student:", question)
    print("Tutor: ", end="")
    pieces = []
    for token in qa_chain.stream(question):
        print(token, end="", flush=True)
        pieces.append(token)
    print("\n")
    answer = "".join(pieces).strip()
    assert answer, "The model returned an empty string."
    return answer


grounded = ask("What is the difference between a relationship, a join, and data blending?")
assert len(grounded) > 40
print("Part 9 stream test passed.")


Student: What is the difference between a relationship, a join, and data blending?
Tutor: **Lecture: Blending Data in Tableau**  

- **Join** – You create it manually in Tableau (or in the database). The two tables are combined at the database level, the aggregation happens there, and only the joined result is returned to Tableau.  

- **Data blending** – Tableau does it automatically while you work on a sheet. The tables remain separate data sources; each source runs its own query, and Tableau aggregates the results in the view, then links them on a common field (a left‑join‑like behavior). Blending also prevents duplicate‑inflated results because aggregation is based on the granularity of the sheet.  

The course excerpts do not discuss **relationships**, so that topic isn’t covered here.

Part 9 stream test passed.


In [ ]:
# API tests. Four student questions, the last one outside the course.
demo_questions = [
    "What is the difference between a dimension and a measure?",
    "How do I build a bar chart of sales by category?",
    "Why should a profit ratio be SUM([Profit]) / SUM([Sales]) rather than the sum of row ratios?",
    "How do I configure a Hadoop cluster for Tableau?",
]

demo_answers = [ask(question) for question in demo_questions]

assert all(demo_answers)
print(f"Answered {len(demo_answers)} questions.")


Student: What is the difference between a dimension and a measure?
Tutor: Dimensions are categorical, qualitative fields that cannot be aggregated, while Measures are numerical, quantitative fields that can be aggregated. (See **Section: Introduction to Tableau – Lecture: Exploring Tableau's interface**, Excerpt 2)

Student: How do I build a bar chart of sales by category?
Tutor: In the **Exploring Tableau's interface** lecture it’s explained that you create a chart by dragging fields onto the Columns and Rows shelves and then choosing the visual type with **Show Me**.  

So, to build a bar chart of sales by category:

1. Drag the **Category** dimension to the **Columns** shelf.  
2. Drag the **Sales** measure to the **Rows** shelf.  
3. Click the **Show Me** button on the right and select the bar‑chart icon.  

This will produce a bar chart showing total sales for each category.

Student: Why should a profit ratio be SUM([Profit]) / SUM([Sales]) rather than the sum of row ratios?
Tuto

## Regression checks

Local checks for the index, the embedding size and the context formatter. No API call.

In [ ]:
assert len(lectures) == EXPECTED_LECTURES
assert len(chunks) > EXPECTED_LECTURES
assert vectorstore._collection.count() == len(chunks)
assert len(embedding.embed_query("tableau workbook")) == EMBEDDING_DIM

regression_hits = retriever.invoke("When do I use the Data Interpreter?")
assert len(regression_hits) == RETRIEVER_K
regression_text = format_context(regression_hits)
assert "Excerpt 1" in regression_text
assert "Lecture:" in regression_text

print("All local regression checks passed.")
print("Index chunks:", len(chunks))
print("Lectures:", ", ".join(titles))


All local regression checks passed.
Index chunks: 136
Lectures: Welcome to Tableau, Why use Tableau: Make your data make an impact, Let's download Tableau Public, Connecting data in Tableau, Exploring Tableau's interface, Let's create our first chart in Tableau!, Creating a table, Creating custom fields, Creating a custom field and adding calculations to a table, Adding totals and subtotals, Adding a custom calculation, Inserting a filter, Working with joins in Tableau, Introduction to the exercise, Let's create a dashboard - Visualizing the three charts we want to create, Using Joins in Tableau, Performing a Numbers Check - Attempt #1, Blending Data in Tableau, Performing a Numbers Check - Attempt #2, First chart, Second chart, Third Chart, Creating and Formatting a Dashboard, Adding Interactive Filters for Improved Analysis, Interactive filters - fix


## Troubleshooting

| Symptom | Fix |
| --- | --- |
| `MarkdownHeaderTextSplitter` returns one document | Join PDF pages with `\n`, not spaces (Part 1). |
| `GROQ_API_KEY` assertion fails | Create `.env` from `.env.example`; the line is `GROQ_API_KEY=...` with no quotes. |
| Answers use old wording after the PDF changes | Delete `data/corrected_lectures.json` and `chroma_intro_tableau/`, then rerun. |
| Chroma count is double the chunk count | The index folder was reused; Part 5 deletes it before rebuilding. |
| Retrieval looks random | Query and stored vectors must use the same embedding model. |

## Swapping in OpenAI

Replace `ChatGroq` with `ChatOpenAI` and `HuggingFaceEmbeddings` with `OpenAIEmbeddings` (package `langchain-openai`, key `OPENAI_API_KEY`). The LCEL chain is unchanged. Rebuild the Chroma index after switching embedding models.